<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Bölüm 2: Metin Verileriyle Çalışmak

Bu not defterinde kullanılan paketler:

In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.5.1
tiktoken version: 0.7.0


- Bu bölüm, girdi verisini LLM için "hazır" hâle getiren veri hazırlama ve örnekleme adımlarını kapsar

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/01.webp?timestamp=1" width="500px">

&nbsp;
## 2.1 Kelime gömmelerini (word embeddings) anlamak

- Bu bölümde kod yok

- Gömmelerin (embedding) pek çok türü vardır; bu kitapta metin gömmelerine odaklanıyoruz

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/02.webp" width="500px">

- LLM'ler gömmelerle yüksek boyutlu uzaylarda çalışır (yani binlerce boyut)
- Bu kadar yüksek boyutlu uzayları görselleştiremediğimiz için (biz insanlar 1, 2 veya 3 boyutta düşünürüz), aşağıdaki şekil 2 boyutlu bir gömme uzayını gösteriyor

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/03.webp" width="300px">

&nbsp;
## 2.2 Metni token'lara ayırmak

- Bu bölümde metni token'lara ayırıyoruz; yani metni tek tek kelimeler ve noktalama işaretleri gibi daha küçük birimlere bölüyoruz

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/04.webp" width="300px">

- Üzerinde çalışmak istediğimiz ham metni yükleyelim
- [Edith Wharton'ın The Verdict](https://en.wikisource.org/wiki/The_Verdict) adlı eseri kamu malı (public domain) bir kısa hikâyedir

In [2]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)


# Kitapta aslında aşağıdaki kod kullanılmıştı
# Ancak urllib, bazı okurların VPN kullanımında sorun
# çıkarabilen eski protokol ayarlarını kullanıyor.
# Yukarıdaki `requests` sürümü bu açıdan
# daha sağlamdır.

"""
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)
"""

<br>

---

<br>

#### SSL sertifika hatalarını giderme

- Bazı okurlar, VSCode veya Jupyter içinde `urllib.request.urlretrieve` çalıştırırken ssl.SSLCertVerificationError: `SSL: CERTIFICATE_VERIFY_FAILED` hatası gördüklerini bildirdi.
- Bu genellikle Python'un sertifika paketinin güncel olmadığı anlamına gelir.


**Çözümler**

- Python ≥ 3.9 kullanın; Python sürümünüzü şu kodu çalıştırarak kontrol edebilirsiniz:
```python
import sys
print(sys.__version__)
```
- Sertifika paketini yükseltin:
  - pip: `pip install --upgrade certifi`
  - uv: `uv pip install --upgrade certifi`
- Yükselttikten sonra Jupyter çekirdeğini (kernel) yeniden başlatın.
- Önceki kod hücresini çalıştırırken hâlâ bir `ssl.SSLCertVerificationError` alıyorsanız, lütfen şuradaki tartışmaya bakın: [GitHub'da daha fazla bilgi](https://github.com/rasbt/LLMs-from-scratch/pull/403)

<br>

---

<br>

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- Amaç, bu metni bir LLM için token'lara ayırmak ve gömmek
- Önce basit bir örnek metin üzerinde basit bir tokenizer geliştirelim; sonra bunu yukarıdaki metne uygulayabiliriz
- Aşağıdaki düzenli ifade (regular expression) boşluklardan bölecek

In [4]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


- Yalnızca boşluklardan değil, virgül ve noktalardan da bölmek istiyoruz; düzenli ifadeyi bunu da yapacak şekilde değiştirelim

In [5]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


- Görüldüğü gibi bu, boş dizeler (empty string) oluşturuyor; onları kaldıralım

In [6]:
# Her öğenin başındaki/sonundaki boşlukları kırp, sonra boş dizeleri ele.
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


- Bu oldukça iyi görünüyor, ama nokta, soru işareti gibi diğer noktalama türlerini de ele alalım

In [7]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


- Bu gayet iyi; artık bu token'lara ayırma işlemini ham metne uygulamaya hazırız

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/05.webp" width="350px">

In [8]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


- Toplam token sayısını hesaplayalım

In [9]:
print(len(preprocessed))

4690


&nbsp;
## 2.3 Token'ları token kimliklerine (token ID) dönüştürmek

- Ardından metin token'larını, daha sonra gömme katmanları üzerinden işleyebileceğimiz token kimliklerine dönüştürüyoruz

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/06.webp" width="500px">

- Bu token'lardan, tüm benzersiz token'lardan oluşan bir sözlük (vocabulary) oluşturabiliriz

In [10]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [11]:
vocab = {token:integer for integer,token in enumerate(all_words)}

- Aşağıda bu sözlükteki ilk 50 kayıt yer alıyor:

In [12]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


- Aşağıda, küçük bir sözlük kullanarak kısa bir örnek metnin token'lara ayrılışını gösteriyoruz:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/07.webp?123" width="500px">

- Şimdi hepsini bir tokenizer sınıfında bir araya getirelim

In [13]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Belirtilen noktalama işaretlerinden önceki boşlukları değiştir
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

- `encode` fonksiyonu metni token kimliklerine çevirir
- `decode` fonksiyonu token kimliklerini tekrar metne çevirir

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="500px">

- Tokenizer'ı, metinleri tam sayılara kodlamak (yani token'lara ayırmak) için kullanabiliriz
- Bu tam sayılar daha sonra LLM'in girdisi olarak gömülebilir

In [14]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


- Tam sayıların kodunu çözüp tekrar metne dönebiliriz

In [15]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [16]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

&nbsp;
## 2.4 Özel bağlam token'ları eklemek

- Bilinmeyen kelimeler için ve bir metnin sonunu belirtmek için bazı "özel" token'lar eklemek faydalıdır

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="500px">

- Bazı tokenizer'lar, LLM'e ek bağlam sağlamak için özel token'lar kullanır
- Bu özel token'lardan bazıları şunlardır
  - `[BOS]` (beginning of sequence, dizi başlangıcı) metnin başlangıcını işaretler
  - `[EOS]` (end of sequence, dizi sonu) metnin nerede bittiğini işaretler (bu genellikle birbiriyle ilgisiz birden çok metni birleştirmek için kullanılır; ör. iki farklı Wikipedia makalesi veya iki farklı kitap gibi)
  - `[PAD]` (padding, dolgu) LLM'leri 1'den büyük bir yığın boyutuyla eğitiyorsak kullanılır (farklı uzunlukta birden çok metin olabilir; dolgu token'ı ile kısa metinleri en uzun olanın uzunluğuna kadar doldururuz, böylece tüm metinler eşit uzunlukta olur)
- `[UNK]` sözlükte bulunmayan kelimeleri temsil eder

- GPT-2'nin yukarıda bahsedilen token'ların hiçbirine ihtiyacı olmadığını, karmaşıklığı azaltmak için yalnızca bir `<|endoftext|>` token'ı kullandığını unutmayın
- `<|endoftext|>`, yukarıda bahsedilen `[EOS]` token'ının karşılığıdır
- GPT ayrıca dolgu için de `<|endoftext|>` kullanır (yığınlanmış girdilerle eğitirken genellikle bir maske kullandığımız için dolgu token'larına zaten dikkat etmeyiz; dolayısıyla bu token'ların ne olduğu önemli değildir)
- GPT-2, sözlük dışı kelimeler için bir `<UNK>` token'ı kullanmaz; bunun yerine kelimeleri alt kelime birimlerine bölen bir byte-pair encoding (BPE) tokenizer'ı kullanır — bunu ilerideki bir bölümde ele alacağız



- Birbirinden bağımsız iki metin kaynağı arasında `<|endoftext|>` token'larını kullanıyoruz:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="500px">

- Aşağıdaki metni token'lara ayırırsak ne olduğuna bakalım:

In [17]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

KeyError: 'Hello'

- Yukarıdaki kod hata veriyor; çünkü "Hello" kelimesi sözlükte yok
- Bu tür durumları ele almak için, bilinmeyen kelimeleri temsil etmek üzere sözlüğe `"<|unk|>"` gibi özel token'lar ekleyebiliriz
- Zaten sözlüğü genişletiyorken, GPT-2 eğitiminde bir metnin sonunu belirtmek için kullanılan `"<|endoftext|>"` adlı bir token daha ekleyelim (bu token, eğitim veri kümemiz birden çok makale, kitap vb. içeriyorsa birleştirilen metinler arasında da kullanılır)

In [18]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [19]:
len(vocab.items())

1132

In [20]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


- Ayrıca tokenizer'ı, yeni `<unk>` token'ını ne zaman ve nasıl kullanacağını bilecek şekilde uyarlamamız gerekiyor

In [21]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Belirtilen noktalama işaretlerinden önceki boşlukları değiştir
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

Değiştirilmiş tokenizer ile metni token'lara ayırmayı deneyelim:

In [22]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [23]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [24]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

&nbsp;
## 2.5 BytePair encoding

- GPT-2, tokenizer olarak BytePair encoding (BPE) kullandı
- Bu, modelin önceden tanımlı sözlüğünde bulunmayan kelimeleri daha küçük alt kelime birimlerine, hatta tek tek karakterlere ayırmasına olanak tanır; böylece sözlük dışı kelimeleri de işleyebilir
- Örneğin, GPT-2'nin sözlüğünde "unfamiliarword" kelimesi yoksa, eğitilmiş BPE birleştirmelerine bağlı olarak bunu ["unfam", "iliar", "word"] ya da başka bir alt kelime ayrımı olarak token'lara ayırabilir
- Orijinal BPE tokenizer'ı burada bulunabilir: [https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- Bu bölümde, çekirdek algoritmalarını hesaplama başarımını artırmak için Rust ile uygulayan OpenAI'ın açık kaynaklı [tiktoken](https://github.com/openai/tiktoken) kütüphanesindeki BPE tokenizer'ını kullanıyoruz
- Bu iki uygulamayı yan yana karşılaştıran bir not defterini [./bytepair_encoder](../02_bonus_bytepair-encoder) klasöründe oluşturdum (tiktoken, örnek metinde yaklaşık 5 kat daha hızlıydı)

In [25]:
# pip install tiktoken

In [26]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.7.0


In [27]:
tokenizer = tiktoken.get_encoding("gpt2")

In [28]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [29]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


- BPE tokenizer'ları bilinmeyen kelimeleri alt kelimelere ve tek tek karakterlere ayırır:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="300px">

## 2.6 Kayan pencere ile veri örnekleme

- LLM'leri her seferinde bir kelime üretecek şekilde eğitiyoruz; bu nedenle eğitim verisini, bir dizideki bir sonraki kelimenin tahmin edilecek hedef olacağı şekilde hazırlamak istiyoruz:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="400px">

In [30]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


- Her metin parçası için girdileri ve hedefleri istiyoruz
- Modelin bir sonraki kelimeyi tahmin etmesini istediğimiz için, hedefler girdilerin bir konum sağa kaydırılmış hâlidir

In [31]:
enc_sample = enc_text[50:]

In [32]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


- Tek tek bakıldığında tahmin şöyle görünür:

In [33]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [34]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


- Bir sonraki kelime tahminini, dikkat (attention) mekanizmasını ele aldıktan sonra ilerideki bir bölümde işleyeceğiz
- Şimdilik, girdi veri kümesi üzerinde dolaşan ve girdilerle bir konum kaydırılmış hedefleri döndüren basit bir veri yükleyici uyguluyoruz

- PyTorch'u kurun ve içe aktarın (kurulum ipuçları için bkz. Ek A)

In [35]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.5.1


- Konumu +1 kaydıran bir kayan pencere (sliding window) yaklaşımı kullanıyoruz:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/13.webp?123" width="500px">

- Girdi metin veri kümesinden parçalar çıkaran bir veri kümesi (dataset) ve veri yükleyici (dataloader) oluşturalım

In [36]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Metnin tamamını token'lara ayır
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Kitabı max_length uzunluğunda örtüşen dizilere bölmek için kayan pencere kullan
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [37]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Tokenizer'ı başlat
    tokenizer = tiktoken.get_encoding("gpt2")

    # Veri kümesini oluştur
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Veri yükleyiciyi oluştur
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

- Bağlam boyutu 4 olan bir LLM için veri yükleyiciyi 1 yığın boyutuyla test edelim:

In [38]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [39]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [40]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


- Adım aralığının (stride) bağlam uzunluğuna eşit olduğu (burada: 4) bir örnek aşağıda gösterilmiştir:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="500px">

- Yığınlanmış çıktılar da oluşturabiliriz
- Burada adım aralığını artırdığımıza dikkat edin; böylece yığınlar arasında örtüşme olmaz — daha fazla örtüşme aşırı öğrenmeyi (overfitting) artırabilir

In [41]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


&nbsp;
## 2.7 Token gömmeleri oluşturmak

- Veri artık bir LLM için neredeyse hazır
- Ama son olarak, bir gömme katmanı kullanarak token'ları sürekli bir vektör temsiline gömelim
- Genellikle bu gömme katmanları LLM'in kendisinin bir parçasıdır ve model eğitimi sırasında güncellenir (eğitilir)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="400px">

- (Token'lara ayırma sonrasında) 2, 3, 5 ve 1 girdi kimliklerine sahip şu dört girdi örneğimiz olduğunu varsayalım:

In [42]:
input_ids = torch.tensor([2, 3, 5, 1])

- Basitlik adına, yalnızca 6 kelimelik küçük bir sözlüğümüz olduğunu ve boyutu 3 olan gömmeler oluşturmak istediğimizi varsayalım:

In [43]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- Bu, 6x3 boyutunda bir ağırlık matrisiyle sonuçlanır:

In [44]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


- One-hot kodlamaya aşina olanlar için: yukarıdaki gömme katmanı yaklaşımı, esasen one-hot kodlamanın ardından tam bağlantılı bir katmanda matris çarpımı yapmanın daha verimli bir uygulanış biçimidir; bu, [./embedding_vs_matmul](../03_bonus_embedding-vs-matmul) klasöründeki tamamlayıcı kodda anlatılmıştır
- Gömme katmanı, one-hot kodlama ve matris çarpımı yaklaşımına eşdeğer olan daha verimli bir uygulama olduğundan, geri yayılım (backpropagation) ile optimize edilebilen bir sinir ağı katmanı olarak görülebilir

- Kimliği 3 olan bir token'ı 3 boyutlu bir vektöre dönüştürmek için şunu yaparız:

In [45]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


- Yukarıdakinin, `embedding_layer` ağırlık matrisindeki 4. satır olduğuna dikkat edin
- Yukarıdaki dört `input_ids` değerinin tamamını gömmek için şunu yaparız

In [46]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


- Gömme katmanı esasen bir arama (look-up) işlemidir:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="500px">

- **Gömme katmanlarını klasik doğrusal katmanlarla karşılaştıran bonus içerik ilginizi çekebilir: [../03_bonus_embedding-vs-matmul](../03_bonus_embedding-vs-matmul)**

&nbsp;
## 2.8 Kelime konumlarını kodlamak

- Gömme katmanı, kimlikleri girdi dizisinde nerede bulunduklarından bağımsız olarak aynı vektör temsillerine dönüştürür:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/17.webp" width="400px">

- Konum gömmeleri (positional embeddings), büyük dil modeline verilecek girdi gömmelerini oluşturmak üzere token gömme vektörüyle birleştirilir:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="500px">

- BytePair encoder'ın sözlük boyutu 50.257'dir:
- Girdi token'larını 256 boyutlu bir vektör temsiline kodlamak istediğimizi varsayalım:

In [47]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- Veri yükleyiciden veri örneklersek, her yığındaki token'ları 256 boyutlu bir vektöre gömeriz
- Her biri 4 token olan 8'lik bir yığın boyutumuz varsa, bu 8 x 4 x 256 boyutunda bir tensör verir:

In [48]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [49]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [50]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# gömmelerin nasıl göründüğünü görmek için aşağıdaki satırı açıp çalıştırın
# print(token_embeddings)

torch.Size([8, 4, 256])


- GPT-2 mutlak konum gömmeleri kullanır; bu yüzden yalnızca bir gömme katmanı daha oluşturuyoruz:

In [51]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# gömme katmanı ağırlıklarının nasıl göründüğünü görmek için aşağıdaki satırı açıp çalıştırın
# print(pos_embedding_layer.weight)

In [52]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# gömmelerin nasıl göründüğünü görmek için aşağıdaki satırı açıp çalıştırın
# print(pos_embeddings)

torch.Size([4, 256])


- Bir LLM'de kullanılan girdi gömmelerini oluşturmak için token gömmeleriyle konum gömmelerini basitçe topluyoruz:

In [53]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# gömmelerin nasıl göründüğünü görmek için aşağıdaki satırı açıp çalıştırın
# print(input_embeddings)

torch.Size([8, 4, 256])


- Girdi işleme akışının ilk aşamasında, girdi metni ayrı token'lara bölünür
- Bu bölmenin ardından, bu token'lar önceden tanımlanmış bir sözlüğe göre token kimliklerine dönüştürülür:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/19.webp" width="400px">

&nbsp;
## Özet ve çıkarımlar

Bu bölümde uyguladığımız veri yükleyicinin derli toplu bir sürümü olan ve ilerideki bölümlerde GPT modelini eğitirken ihtiyaç duyacağımız [./dataloader.ipynb](./dataloader.ipynb) kod not defterine bakın.

Alıştırma çözümleri için [./exercise-solutions.ipynb](./exercise-solutions.ipynb) dosyasına bakın.

GPT-2 tokenizer'ının sıfırdan nasıl uygulanıp eğitilebileceğini öğrenmek istiyorsanız [Sıfırdan Byte Pair Encoding (BPE) Tokenizer](../02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb) not defterine bakın.